In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from itertools import product

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import (
    RandomForestClassifier, 
    ExtraTreesClassifier, 
    GradientBoostingClassifier, 
    AdaBoostClassifier
)

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.utils import resample
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import classification_report, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

In [3]:
df = pd.read_csv('base-limpia.csv')

In [4]:
# Separar target de features
target = 'fraud_bool'
X = df.drop(columns=[target])
y = df[target]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:
print("Iniciando pipeline de entrenamiento y evaluación...")
print("-" * 60)

# =====================================================================
# 0. CREAR SUBCONJUNTO ESTRATIFICADO (10% de Train para GridSearch)
# =====================================================================
# Esto mantiene la proporción exacta de fraudes (0 y 1) pero usa menos datos para ser rápido
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.90, random_state=42)
for train_idx_small, _ in sss.split(X_train, y_train):
    X_train_small = X_train.iloc[train_idx_small]
    y_train_small = y_train.iloc[train_idx_small]

print(f"Tamaño Train Original: {X_train.shape[0]} muestras.")
print(f"Tamaño Train Submuestreado (10%): {X_train_small.shape[0]} muestras para GridSearch.\n")


Iniciando pipeline de entrenamiento y evaluación...
------------------------------------------------------------
Tamaño Train Original: 800000 muestras.
Tamaño Train Submuestreado (10%): 80000 muestras para GridSearch.



In [ ]:



# =====================================================================
# 1. SVM LINEAL (Optimizado para datasets masivos)
# =====================================================================
print("Entrenando LinearSVC...")
start = time.time()
pipe_svc = Pipeline([
    ('scaler', StandardScaler()), 
    ('svm', LinearSVC(random_state=42, class_weight='balanced', dual='auto', max_iter=5000))
])

param_svc = {'svm__C': [0.01, 0.1, 1, 10]}
grid_svc = GridSearchCV(pipe_svc, param_svc, cv=5, scoring='average_precision', n_jobs=-1)
grid_svc.fit(X_train_small, y_train_small)

print(f"Mejor SVM: {grid_svc.best_params_}")
print(f"AP Score (CV): {grid_svc.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 2. ÁRBOL DE DECISIÓN (Con Poda)
# =====================================================================
print("Entrenando Decision Tree...")
start = time.time()
tree = DecisionTreeClassifier(random_state=42, class_weight='balanced')

param_tree = {
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'ccp_alpha': [0.0, 0.001, 0.01]
}
grid_tree = GridSearchCV(tree, param_tree, cv=5, scoring='average_precision', n_jobs=-1)
grid_tree.fit(X_train_small, y_train_small)

print(f"Mejor Árbol: {grid_tree.best_params_}")
print(f"AP Score (CV): {grid_tree.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 3. RANDOM FOREST
# =====================================================================
print("Entrenando Random Forest...")
start = time.time()
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5]
}
grid_rf = GridSearchCV(rf, param_rf, cv=5, scoring='average_precision', n_jobs=-1)
grid_rf.fit(X_train_small, y_train_small)

print(f"Mejor RF: {grid_rf.best_params_}")
print(f"AP Score (CV): {grid_rf.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4. GRADIENT BOOSTING (Con manejo de pesos manual)
# =====================================================================
print("Entrenando Gradient Boosting...")
start = time.time()
gb = GradientBoostingClassifier(random_state=42)

param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

# Calculamos los pesos para pasarlos al GB, ya que no tiene class_weight nativo
pesos_train_small = compute_sample_weight(class_weight='balanced', y=y_train_small)

grid_gb = GridSearchCV(gb, param_gb, cv=5, scoring='average_precision', n_jobs=-1)
# Pasamos los pesos a través del parámetro fit_params
grid_gb.fit(X_train_small, y_train_small, sample_weight=pesos_train_small)

print(f"Mejor GB: {grid_gb.best_params_}")
print(f"AP Score (CV): {grid_gb.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 5. EVALUACIÓN FINAL: ENTRENAR EL MEJOR MODELO CON EL 100% DE LOS DATOS
# =====================================================================
print("-" * 60)
print("SELECCIONANDO EL MEJOR MODELO GENERAL...")

# Comparamos el score AP de todos los grids para encontrar al ganador absoluto
resultados = {
    "LinearSVC": (grid_svc.best_score_, grid_svc.best_estimator_),
    "DecisionTree": (grid_tree.best_score_, grid_tree.best_estimator_),
    "RandomForest": (grid_rf.best_score_, grid_rf.best_estimator_),
    "GradientBoosting": (grid_gb.best_score_, grid_gb.best_estimator_)
}

# Encontrar el modelo con el mejor AP Score
nombre_mejor_modelo = max(resultados, key=lambda k: resultados[k][0])
mejor_modelo_final = resultados[nombre_mejor_modelo][1]

print(f"¡El modelo ganador es {nombre_mejor_modelo} con un AP Score en CV de {resultados[nombre_mejor_modelo][0]:.4f}!")
print("Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...")

# Reentrenamos con el 100% de Train (incluyendo manejo de pesos si es GB)
start_final = time.time()
if nombre_mejor_modelo == "GradientBoosting":
    pesos_train_full = compute_sample_weight(class_weight='balanced', y=y_train)
    mejor_modelo_final.fit(X_train, y_train, sample_weight=pesos_train_full)
else:
    mejor_modelo_final.fit(X_train, y_train)
    
print(f"Entrenamiento final completado en {time.time()-start_final:.2f}s.\n")

# =====================================================================
# 6. MÉTRICAS EN TEST
# =====================================================================
print(f"--- REPORTE DE CLASIFICACIÓN EN TEST ({nombre_mejor_modelo}) ---")

# Predecir clases e inferir probabilidades
y_pred = mejor_modelo_final.predict(X_test)

# Excepción para LinearSVC que usa decision_function en lugar de predict_proba
if nombre_mejor_modelo == "LinearSVC":
    y_proba = mejor_modelo_final.decision_function(X_test)
else:
    y_proba = mejor_modelo_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
ap_test = average_precision_score(y_test, y_proba)
print("-" * 60)
print(f"Average Precision (AP) Final en Test: {ap_test:.4f}")
print("-" * 60)

Iniciando pipeline de entrenamiento y evaluación...
------------------------------------------------------------
Tamaño Train Original: 800000 muestras.
Tamaño Train Submuestreado (10%): 80000 muestras para GridSearch.

Entrenando LinearSVC...
Mejor SVM: {'svm__C': 0.01}
AP Score (CV): 0.1368 | Tiempo: 26.42s

Entrenando Decision Tree...
Mejor Árbol: {'ccp_alpha': 0.001, 'max_depth': 20, 'min_samples_leaf': 20}
AP Score (CV): 0.0480 | Tiempo: 40.25s

Entrenando Random Forest...
Mejor RF: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
AP Score (CV): 0.1255 | Tiempo: 102.93s

Entrenando Gradient Boosting...
Mejor GB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
AP Score (CV): 0.8724 | Tiempo: 282.97s

------------------------------------------------------------
SELECCIONANDO EL MEJOR MODELO GENERAL...
¡El modelo ganador es GradientBoosting con un AP Score en CV de 0.8724!
Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...


KeyboardInterrupt: 

In [8]:
# =====================================================================
# EXTRA. ADABOOST
# =====================================================================
print("Entrenando AdaBoost...")
start = time.time()

# Para que AdaBoost entienda el desbalance, le pasamos un árbol base "balanceado"
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')

ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)

param_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

grid_ada = GridSearchCV(ada, param_ada, cv=10, scoring='average_precision', n_jobs=-1)
grid_ada.fit(X_train_small, y_train_small)

print(f"Mejor AdaBoost: {grid_ada.best_params_}")
print(f"AP Score (CV): {grid_ada.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

Entrenando AdaBoost...
Mejor AdaBoost: {'learning_rate': 0.1, 'n_estimators': 50}
AP Score (CV): 0.0193 | Tiempo: 11.12s



In [9]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.metrics import classification_report, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

print("Iniciando pipeline de entrenamiento y evaluación...")
print("-" * 60)

# =====================================================================
# 0. CREAR SUBCONJUNTO ESTRATIFICADO (10% de Train para GridSearch)
# =====================================================================
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.90, random_state=42)
for train_idx_small, _ in sss.split(X_train, y_train):
    X_train_small = X_train.iloc[train_idx_small]
    y_train_small = y_train.iloc[train_idx_small]

print(f"Tamaño Train Original: {X_train.shape[0]} muestras.")
print(f"Tamaño Train Submuestreado (10%): {X_train_small.shape[0]} muestras para GridSearch.\n")

# =====================================================================
# 1. SVM LINEAL
# =====================================================================
print("Entrenando LinearSVC...")
start = time.time()
pipe_svc = Pipeline([
    ('scaler', StandardScaler()), 
    ('svm', LinearSVC(random_state=42, class_weight='balanced', dual='auto', max_iter=5000))
])

param_svc = {'svm__C': [0.01, 0.1, 1, 10]}
grid_svc = GridSearchCV(pipe_svc, param_svc, cv=5, scoring='average_precision', n_jobs=-1)
grid_svc.fit(X_train_small, y_train_small)

print(f"Mejor SVM: {grid_svc.best_params_}")
print(f"AP Score (CV): {grid_svc.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 2. ÁRBOL DE DECISIÓN
# =====================================================================
print("Entrenando Decision Tree...")
start = time.time()
tree = DecisionTreeClassifier(random_state=42, class_weight='balanced')

param_tree = {
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'ccp_alpha': [0.0, 0.001, 0.01]
}
grid_tree = GridSearchCV(tree, param_tree, cv=5, scoring='average_precision', n_jobs=-1)
grid_tree.fit(X_train_small, y_train_small)

print(f"Mejor Árbol: {grid_tree.best_params_}")
print(f"AP Score (CV): {grid_tree.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 3. RANDOM FOREST
# =====================================================================
print("Entrenando Random Forest...")
start = time.time()
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5]
}
grid_rf = GridSearchCV(rf, param_rf, cv=5, scoring='average_precision', n_jobs=-1)
grid_rf.fit(X_train_small, y_train_small)

print(f"Mejor RF: {grid_rf.best_params_}")
print(f"AP Score (CV): {grid_rf.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4. GRADIENT BOOSTING
# =====================================================================
print("Entrenando Gradient Boosting...")
start = time.time()
gb = GradientBoostingClassifier(random_state=42)

param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

pesos_train_small = compute_sample_weight(class_weight='balanced', y=y_train_small)
grid_gb = GridSearchCV(gb, param_gb, cv=5, scoring='average_precision', n_jobs=-1)
grid_gb.fit(X_train_small, y_train_small, sample_weight=pesos_train_small)

print(f"Mejor GB: {grid_gb.best_params_}")
print(f"AP Score (CV): {grid_gb.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 4.5 ADABOOST
# =====================================================================
print("Entrenando AdaBoost...")
start = time.time()
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42, class_weight='balanced')
ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)

param_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.1, 0.5, 1.0]
}

grid_ada = GridSearchCV(ada, param_ada, cv=5, scoring='average_precision', n_jobs=-1)
grid_ada.fit(X_train_small, y_train_small)

print(f"Mejor AdaBoost: {grid_ada.best_params_}")
print(f"AP Score (CV): {grid_ada.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

# =====================================================================
# 5. SELECCIÓN DEL MEJOR MODELO Y REENTRENAMIENTO (100% DATOS)
# =====================================================================
print("-" * 60)
print("SELECCIONANDO EL MEJOR MODELO GENERAL...")

resultados = {
    "LinearSVC": (grid_svc.best_score_, grid_svc.best_estimator_),
    "DecisionTree": (grid_tree.best_score_, grid_tree.best_estimator_),
    "RandomForest": (grid_rf.best_score_, grid_rf.best_estimator_),
    "GradientBoosting": (grid_gb.best_score_, grid_gb.best_estimator_),
    "AdaBoost": (grid_ada.best_score_, grid_ada.best_estimator_) # Añadido AdaBoost aquí
}

nombre_mejor_modelo = max(resultados, key=lambda k: resultados[k][0])
mejor_modelo_final = resultados[nombre_mejor_modelo][1]

print(f"¡El modelo ganador es {nombre_mejor_modelo} con un AP Score en CV de {resultados[nombre_mejor_modelo][0]:.4f}!")
print("Reentrenando el modelo ganador con el 100% de los datos de entrenamiento...")

start_final = time.time()
if nombre_mejor_modelo == "GradientBoosting":
    pesos_train_full = compute_sample_weight(class_weight='balanced', y=y_train)
    mejor_modelo_final.fit(X_train, y_train, sample_weight=pesos_train_full)
else:
    mejor_modelo_final.fit(X_train, y_train)
    
print(f"Entrenamiento final completado en {time.time()-start_final:.2f}s.\n")

# =====================================================================
# 6. MÉTRICAS EN TEST
# =====================================================================
print(f"--- REPORTE DE CLASIFICACIÓN EN TEST ({nombre_mejor_modelo}) ---")

y_pred = mejor_modelo_final.predict(X_test)

if nombre_mejor_modelo == "LinearSVC":
    y_proba = mejor_modelo_final.named_steps['svm'].decision_function(X_test)
else:
    y_proba = mejor_modelo_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
ap_test = average_precision_score(y_test, y_proba)
print("-" * 60)
print(f"Average Precision (AP) Final en Test: {ap_test:.4f}")
print("-" * 60)

# =====================================================================
# 7. EXTRA: GRÁFICO DE IMPORTANCIA DE VARIABLES (FEATURE IMPORTANCE)
# =====================================================================
# Este bloque enriquecerá muchísimo tu EDA y conclusiones
print("Generando gráfico de Importancia de Variables...")

# Comprobamos si el modelo está en un Pipeline (como LinearSVC) o es un modelo directo
modelo_para_importancias = mejor_modelo_final.named_steps['svm'] if nombre_mejor_modelo == "LinearSVC" else mejor_modelo_final

if hasattr(modelo_para_importancias, 'feature_importances_'):
    importancias = modelo_para_importancias.feature_importances_
elif hasattr(modelo_para_importancias, 'coef_'):
    importancias = np.abs(modelo_para_importancias.coef_[0]) # Tomamos el valor absoluto para SVM Lineal
else:
    importancias = None

if importancias is not None:
    df_importancias = pd.DataFrame({
        'Variable': X_train.columns,
        'Importancia': importancias
    }).sort_values(by='Importancia', ascending=False).head(15) # Top 15 variables
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importancia', y='Variable', data=df_importancias, palette='viridis')
    plt.title(f'Top 15 Variables más Importantes - {nombre_mejor_modelo}')
    plt.xlabel('Importancia (Magnitud)')
    plt.ylabel('Variable')
    plt.tight_layout()
    plt.show()
else:
    print("El modelo seleccionado no soporta la extracción directa de importancia de variables.")

Iniciando pipeline de entrenamiento y evaluación...
------------------------------------------------------------
Tamaño Train Original: 800000 muestras.
Tamaño Train Submuestreado (10%): 80000 muestras para GridSearch.

Entrenando LinearSVC...
Mejor SVM: {'svm__C': 0.01}
AP Score (CV): 0.1368 | Tiempo: 10.92s

Entrenando Decision Tree...
Mejor Árbol: {'ccp_alpha': 0.001, 'max_depth': 20, 'min_samples_leaf': 20}
AP Score (CV): 0.0480 | Tiempo: 35.13s

Entrenando Random Forest...
Mejor RF: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
AP Score (CV): 0.1255 | Tiempo: 90.95s

Entrenando Gradient Boosting...
Mejor GB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
AP Score (CV): 0.8724 | Tiempo: 279.55s

Entrenando AdaBoost...
Mejor AdaBoost: {'learning_rate': 0.1, 'n_estimators': 50}
AP Score (CV): 0.0192 | Tiempo: 6.10s

------------------------------------------------------------
SELECCIONANDO EL MEJOR MODELO GENERAL...
¡El modelo ganador es GradientBoostin

KeyboardInterrupt: 

In [7]:
from xgboost import XGBClassifier

# =====================================================================
# XGBOOST (Reemplaza o complementa a Gradient Boosting)
# =====================================================================
print("Entrenando XGBoost...")
start = time.time()

xgb = XGBClassifier(
    random_state=42,
    scale_pos_weight=len(y_train_small[y_train_small==0]) / len(y_train_small[y_train_small==1]),  # Crucial!
    eval_metric='aucpr',  # Average Precision como métrica
    use_label_encoder=False,
    n_jobs=-1
)

param_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(xgb, param_xgb, cv=5, scoring='average_precision', n_jobs=-1)
grid_xgb.fit(X_train_small, y_train_small)

print(f"Mejor XGBoost: {grid_xgb.best_params_}")
print(f"AP Score (CV): {grid_xgb.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

Entrenando XGBoost...


c:\Users\pausa\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:33:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Mejor XGBoost: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
AP Score (CV): 0.1497 | Tiempo: 95.48s



In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    random_state=42,
    class_weight='balanced',
    is_unbalance=True,  # Alternativa a class_weight
    n_jobs=-1
)

param_lgbm = {
    'n_estimators': [100, 200],
    'num_leaves': [31, 50, 100],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgbm = GridSearchCV(lgbm, param_lgbm, cv=5, scoring='average_precision', n_jobs=-1)
grid_lgbm.fit(X_train_small, y_train_small)



[LightGBM] [Info] Number of positive: 882, number of negative: 79118
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003419 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3227
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200], 'num_leaves': [31, 50, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fo

In [9]:
print(f"Mejor XGBoost: {grid_lgbm.best_params_}")
print(f"AP Score (CV): {grid_lgbm.best_score_:.4f} | Tiempo: {time.time()-start:.2f}s\n")

Mejor XGBoost: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'n_estimators': 100, 'num_leaves': 31, 'subsample': 0.8}
AP Score (CV): 0.1241 | Tiempo: 393.94s



In [11]:
ratio = len(y_train_small[y_train_small==0]) / len(y_train_small[y_train_small==1])
print(f"Ratio calculado: {ratio:.2f}")
# Debería ser ~99 si tienes 1% de fraudes
# Si el ratio es muy bajo (<10), el problema es otro
# Entrena un solo XGBoost con los mejores parámetros estimados
xgb_test = XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    eval_metric='aucpr'
)

xgb_test.fit(X_train_small, y_train_small)

# Predice en TEST (no en CV)
y_proba_test = xgb_test.predict_proba(X_test)[:, 1]

# Calcula Average Precision manualmente
from sklearn.metrics import average_precision_score
ap_manual = average_precision_score(y_test, y_proba_test)

print(f"AP en TEST (manual): {ap_manual:.4f}")

# Además, mira las primeras predicciones
print("\nPrimeras 10 probabilidades predichas para clase 1 (fraude):")
print(y_proba_test[:10])
print("\nPrimeras 10 etiquetas reales:")
print(y_test[:10].values)

# Si las probabilidades son todas cercanas a 0 (ej. 0.001, 0.002), el modelo no está aprendiendo
# Si hay variedad (ej. 0.1, 0.7, 0.3), entonces el modelo SÍ funciona y el problema es de GridSearchCV

Ratio calculado: 89.70
AP en TEST (manual): 0.1520

Primeras 10 probabilidades predichas para clase 1 (fraude):
[0.18036614 0.08643188 0.2579723  0.29287905 0.10982721 0.40777838
 0.78434056 0.13147505 0.5281     0.06893714]

Primeras 10 etiquetas reales:
[0 0 0 0 0 0 0 0 0 0]


In [13]:
import time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score
from sklearn.calibration import CalibratedClassifierCV

print("=" * 70)
print("OPTIMIZACIÓN EXTREMA PARA DATASET DESBALANCEADO")
print("=" * 70)

# =====================================================================
# 0. PREPARACIÓN: Usar SUBSET MÁS GRANDE (30% en lugar de 10%)
# =====================================================================
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=42)  # 30% para entrenar
for train_idx_small, _ in sss.split(X_train, y_train):
    X_train_opt = X_train.iloc[train_idx_small]
    y_train_opt = y_train.iloc[train_idx_small]

print(f"\n📊 Configuración optimizada:")
print(f"   Train size: {len(X_train_opt):,} muestras")
print(f"   Fraudes en train: {y_train_opt.sum():,} ({y_train_opt.mean()*100:.2f}%)")
print(f"   Ratio desbalance: {(y_train_opt==0).sum() / (y_train_opt==1).sum():.1f}:1")

ratio = (y_train_opt == 0).sum() / (y_train_opt == 1).sum()

# =====================================================================
# 1. XGBOOST OPTIMIZADO (con parámetros para desbalance extremo)
# =====================================================================
print("\n" + "=" * 70)
print("1. ENTRENANDO XGBOOST OPTIMIZADO")
print("=" * 70)
start = time.time()

xgb_opt = XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=500,  # Más árboles
    max_depth=4,
    learning_rate=0.01,  # Learning rate más bajo
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=10,  # Regularización extra
    gamma=0.5,  # Reducción mínima de pérdida para dividir nodo
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=1.0,  # L2 regularization
    random_state=42,
    eval_metric='aucpr',
    use_label_encoder=False,
    n_jobs=-1,
    verbosity=0
)

# GridSearch más reducido para tiempo razonable
param_xgb = {
    'learning_rate': [0.01, 0.05],
    'max_depth': [3, 4, 5],
    'min_child_weight': [5, 10, 20]
}

grid_xgb = GridSearchCV(
    xgb_opt, param_xgb, 
    cv=3,  # Menos folds para más rapidez
    scoring='average_precision', 
    n_jobs=-1,
    verbose=0
)

grid_xgb.fit(X_train_opt, y_train_opt)

print(f"✅ Mejor XGBoost: {grid_xgb.best_params_}")
print(f"   AP Score (CV 3-fold): {grid_xgb.best_score_:.4f}")
print(f"   Tiempo: {time.time()-start:.2f}s")

# =====================================================================
# 2. LIGHTGBM OPTIMIZADO (con parámetros para desbalance extremo)
# =====================================================================
print("\n" + "=" * 70)
print("2. ENTRENANDO LIGHTGBM OPTIMIZADO")
print("=" * 70)
start = time.time()

lgb_opt = lgb.LGBMClassifier(
    scale_pos_weight=ratio,
    class_weight='balanced',
    n_estimators=500,
    learning_rate=0.01,
    num_leaves=31,
    max_depth=5,
    min_child_samples=20,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

param_lgb = {
    'learning_rate': [0.01, 0.05],
    'num_leaves': [15, 31, 63],
    'max_depth': [3, 5, 7]
}

grid_lgb = GridSearchCV(
    lgb_opt, param_lgb, 
    cv=3,
    scoring='average_precision', 
    n_jobs=-1,
    verbose=0
)

grid_lgb.fit(X_train_opt, y_train_opt)

print(f"✅ Mejor LightGBM: {grid_lgb.best_params_}")
print(f"   AP Score (CV 3-fold): {grid_lgb.best_score_:.4f}")
print(f"   Tiempo: {time.time()-start:.2f}s")

# =====================================================================
# 3. EVALUACIÓN EN TEST CON THRESHOLD OPTIMIZADO
# =====================================================================
print("\n" + "=" * 70)
print("3. EVALUACIÓN EN TEST CON THRESHOLD OPTIMIZADO")
print("=" * 70)

# Función para evaluar con threshold óptimo
def evaluate_with_threshold(model, X_test, y_test, model_name):
    # Obtener probabilidades
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min())  # Normalizar a [0,1]
    
    # Encontrar threshold óptimo usando validación (20% de train como val)
    from sklearn.model_selection import train_test_split
    X_val, _, y_val, _ = train_test_split(X_train_opt, y_train_opt, test_size=0.8, random_state=42, stratify=y_train_opt)
    
    if hasattr(model, 'predict_proba'):
        y_proba_val = model.predict_proba(X_val)[:, 1]
    else:
        y_proba_val = model.decision_function(X_val)
        y_proba_val = (y_proba_val - y_proba_val.min()) / (y_proba_val.max() - y_proba_val.min())
    
    # Buscar threshold que maximiza F1
    precision, recall, thresholds = precision_recall_curve(y_val, y_proba_val)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores[:-1])  # Excluir último
    best_threshold = thresholds[best_idx] if len(thresholds) > best_idx else 0.5
    
    # Aplicar threshold óptimo
    y_pred_opt = (y_proba >= best_threshold).astype(int)
    
    # Métricas
    ap_raw = average_precision_score(y_test, y_proba)
    ap_opt = average_precision_score(y_test, y_pred_opt)
    f1_opt = f1_score(y_test, y_pred_opt)
    
    print(f"\n📈 {model_name}:")
    print(f"   Threshold óptimo: {best_threshold:.4f}")
    print(f"   AP (probabilidades raw): {ap_raw:.4f}")
    print(f"   AP (con threshold óptimo): {ap_opt:.4f}")
    print(f"   F1-score (threshold óptimo): {f1_opt:.4f}")
    
    return {'ap_raw': ap_raw, 'ap_opt': ap_opt, 'f1': f1_opt, 'threshold': best_threshold, 'proba': y_proba}

# Evaluar ambos modelos
results_xgb = evaluate_with_threshold(grid_xgb.best_estimator_, X_test, y_test, "XGBoost")
results_lgb = evaluate_with_threshold(grid_lgb.best_estimator_, X_test, y_test, "LightGBM")

# =====================================================================
# 4. ENSEMBLE: MEDIA DE PROBABILIDADES (MÁS POTENTE QUE MODELOS INDIVIDUALES)
# =====================================================================
print("\n" + "=" * 70)
print("4. ENSEMBLE (XGBoost + LightGBM)")
print("=" * 70)

# Promediar probabilidades
y_proba_ensemble = (results_xgb['proba'] + results_lgb['proba']) / 2

# Encontrar threshold óptimo para el ensemble
X_val_ensemble, _, y_val_ensemble, _ = train_test_split(X_train_opt, y_train_opt, test_size=0.8, random_state=42, stratify=y_train_opt)
y_proba_val_xgb = grid_xgb.best_estimator_.predict_proba(X_val_ensemble)[:, 1]
y_proba_val_lgb = grid_lgb.best_estimator_.predict_proba(X_val_ensemble)[:, 1]
y_proba_val_ensemble = (y_proba_val_xgb + y_proba_val_lgb) / 2

precision, recall, thresholds = precision_recall_curve(y_val_ensemble, y_proba_val_ensemble)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores[:-1])
best_threshold_ensemble = thresholds[best_idx] if len(thresholds) > best_idx else 0.5

y_pred_ensemble = (y_proba_ensemble >= best_threshold_ensemble).astype(int)
ap_ensemble = average_precision_score(y_test, y_proba_ensemble)
f1_ensemble = f1_score(y_test, y_pred_ensemble)

print(f"📈 Ensemble (promedio simple):")
print(f"   Threshold óptimo: {best_threshold_ensemble:.4f}")
print(f"   AP (probabilidades raw): {ap_ensemble:.4f}")
print(f"   F1-score (threshold óptimo): {f1_ensemble:.4f}")

# =====================================================================
# 5. COMPARATIVA FINAL
# =====================================================================
print("\n" + "=" * 70)
print("5. COMPARATIVA FINAL DE MODELOS EN TEST")
print("=" * 70)

comparativa = pd.DataFrame({
    'Modelo': ['XGBoost', 'LightGBM', 'Ensemble (XGB+LGB)'],
    'AP (raw)': [results_xgb['ap_raw'], results_lgb['ap_raw'], ap_ensemble],
    'AP (threshold opt)': [results_xgb['ap_opt'], results_lgb['ap_opt'], ap_ensemble],
    'F1 (threshold opt)': [results_xgb['f1'], results_lgb['f1'], f1_ensemble]
})

print(comparativa.to_string(index=False))

print("\n" + "=" * 70)
print("🎯 CONCLUSIÓN")
print("=" * 70)
print(f"✅ Mejor AP raw: {max(results_xgb['ap_raw'], results_lgb['ap_raw'], ap_ensemble):.4f}")
print(f"✅ Mejor F1-score: {max(results_xgb['f1'], results_lgb['f1'], f1_ensemble):.4f}")

if ap_ensemble > max(results_xgb['ap_raw'], results_lgb['ap_raw']):
    print("\n🏆 EL ENSEMBLE ES EL MEJOR MODELO")
    print("   La combinación de XGBoost + LightGBM supera a los individuales")

OPTIMIZACIÓN EXTREMA PARA DATASET DESBALANCEADO

📊 Configuración optimizada:
   Train size: 240,000 muestras
   Fraudes en train: 2,647 (1.10%)
   Ratio desbalance: 89.7:1

1. ENTRENANDO XGBOOST OPTIMIZADO
✅ Mejor XGBoost: {'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 20}
   AP Score (CV 3-fold): 0.1702
   Tiempo: 97.05s

2. ENTRENANDO LIGHTGBM OPTIMIZADO
✅ Mejor LightGBM: {'learning_rate': 0.05, 'max_depth': 3, 'num_leaves': 15}
   AP Score (CV 3-fold): 0.1543
   Tiempo: 128.44s

3. EVALUACIÓN EN TEST CON THRESHOLD OPTIMIZADO

📈 XGBoost:
   Threshold óptimo: 0.9080
   AP (probabilidades raw): 0.1735
   AP (con threshold óptimo): 0.0679
   F1-score (threshold óptimo): 0.2425

📈 LightGBM:
   Threshold óptimo: 0.9986
   AP (probabilidades raw): 0.1637
   AP (con threshold óptimo): 0.0638
   F1-score (threshold óptimo): 0.2316

4. ENSEMBLE (XGBoost + LightGBM)
📈 Ensemble (promedio simple):
   Threshold óptimo: 0.9539
   AP (probabilidades raw): 0.1734
   F1-score (threshold 